# Knowledge-Graph Anomaly Detection on Google Colab

This notebook runs the [`pipeline.py`](pipeline.py) end-to-end on a **free GPU** so
training and evaluation finish in a fraction of the CPU run time.

## Before you start: enable the GPU

Go to **Runtime → Change runtime type → Hardware accelerator** and pick **GPU**
(any of T4/L4/A100 works). PyKEEN auto-detects CUDA, so no code changes are
needed – the pipeline will use the GPU automatically once one is attached.

Run the cells below in order.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repository

Replace the URL if you are working from a fork.

In [ ]:
%cd /content
![ -d knowledge-graphs-project ] || git clone https://github.com/arslansmajevic/knowledge-graphs-project.git
%cd knowledge-graphs-project

## 3. Install the dependencies

PyKEEN pulls in a CUDA-enabled PyTorch automatically on Colab.

In [ ]:
!pip install -q -r requirements.txt

## 4. Provide the dataset

The raw LANL logs are **not** committed to the repo (they are far too large), so
you need to make them available to Colab. The pipeline expects them in a
`dataset/` directory at the repo root:

```
dataset/
├── auth.txt
├── proc.txt
├── flows.txt
├── dns.txt
└── redteam.txt
```

The easiest option is to keep the files in **Google Drive** and point the repo's
`dataset/` at them. Upload your (optionally gzipped) LANL files to a folder in
Drive once, then run the cell below and adjust `DRIVE_DATASET_DIR` to match.

> Colab sessions are ephemeral – anything not in Drive is lost when the runtime
> recycles, so storing the dataset in Drive saves you from re-uploading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Folder in your Drive that contains auth.txt, proc.txt, flows.txt, dns.txt, redteam.txt
DRIVE_DATASET_DIR = '/content/drive/MyDrive/lanl-dataset'

# Symlink it as ./dataset so the pipeline finds the files unchanged.
if os.path.islink('dataset') or os.path.exists('dataset'):
    print('dataset/ already present:', os.listdir('dataset'))
else:
    os.symlink(DRIVE_DATASET_DIR, 'dataset')
    print('Linked dataset ->', DRIVE_DATASET_DIR)
    print(os.listdir('dataset'))

<details>
<summary>Alternative: upload the files directly into this session</summary>

If you would rather not use Drive, create the folder and upload the files into
it (they will be lost when the session ends):

```python
import os
from google.colab import files
os.makedirs('dataset', exist_ok=True)
%cd dataset
files.upload()   # select auth.txt, proc.txt, flows.txt, dns.txt, redteam.txt
%cd ..
```
</details>

## 5. Run the pipeline

This runs all four steps (build → train → score → evaluate). With a GPU attached
PyKEEN trains and evaluates on CUDA automatically.

Model and speed settings (which models to train, `EMBEDDING_DIM`, `EPOCHS`,
`MAX_TIME`, `EVAL_SAMPLE_SIZE`, ...) are constants at the top of `pipeline.py`;
edit them there if you want to train on more data or for longer now that you
have a GPU.

In [ ]:
!python pipeline.py

### Running individual steps

Once the graph is built you can iterate on models without rebuilding it:

```python
!python pipeline.py --steps build
!python pipeline.py --steps train score evaluate
```

## 6. (Optional) Keep the results

Artifacts are written to `generated-files/` and trained models to
`pykeen-lanl-model/<model>/`. Copy them to Drive so they survive the session.

In [ ]:
!mkdir -p /content/drive/MyDrive/kg-results
!cp -r generated-files pykeen-lanl-model /content/drive/MyDrive/kg-results/ 2>/dev/null
print('Copied results to /content/drive/MyDrive/kg-results')